# General Overview of making OPENMC models of Nuclear Reactors

## Used for Cardinal Multiphysics simulation

### Goal

Build a Basic Understanding of how to use the OPENMC Python API. This tutorial uses the example of creating a basic Mono-block HTGR core.

**Note:** This tutorial assumes you understand the OpenMC architecture, and have OpenMC built through Cardinal.<br> If you have it built another way, you will need to modify the code in the setup section. <br>If you don't have it built, you can still look at the pre-provided outputs, but you will not be able to run any of the code sections

### Learning Objectives

&nbsp;1. Understand how to setup OPENMC Python API

&nbsp;2. Learn how to define OPENMC materials and properties in python

&nbsp;3. Learn how to define OPENMC geometries in python

&nbsp;4. Learn how to export those files into formats usable by further tools

# Setup
**Note:** This tutorial assumes you have a built OPENMC binary, someway, somehow. 
## Packages
We need to ensure any needed packages are all set for python. First, run the following code section, you can ignore anything about a version error

Expect output in this format:

word soup

0.16.1.dev14+g150f9ac04

<module 'openmc.lib' from '/home/jmiller/NEP/SSTA-NEP/.venv/lib/python3.14/site-packages/openmc/lib/__init__.py'>


In [ ]:

!python -m pip install pip-review
!pip-review --auto
!python -m pip install openmc-plotter
import openmc
import openmc.lib
print(openmc.__version__)
print(openmc.lib)


If that code block runs without errors, you can keep going. If it does not, execute the following code block. It will gather the OpenMC Python Module

*Note: Whenever OpenMC is updated by cardinal in the future, it will require rebuilding the module, which you can do the short way with this code cell*

In [ ]:

%%bash
cd ${NEKRS_HOME%/*}/contrib/openmc
git fetch --tags
python -m pip install setuptools
python -m pip install numpy
python -m pip install .
python -c "import openmc; import openmc.lib; print(openmc.__version__); print(openmc.lib)"



You should expect this output style from the above:

word soup

0.16.1.dev14+g150f9ac04

<module 'openmc.lib' from '/home/jmiller/NEP/SSTA-NEP/.venv/lib/python3.14/site-packages/openmc/lib/__init__.py'>

# Functions
Any neccessary imports and other setup.

In [35]:
import openmc
import numpy as np
import os

# 1. OpenMC Basic Tenants

## &nbsp;A. Structure


&nbsp;What goes into an OpenMC neutronics model?

### &nbsp;&nbsp;1. Geometry

&nbsp;&nbsp;&nbsp;Pretty Self-explanatory. Basically the actual structure of everything. OpenMC uses **CSG Models**, which is just a different way to go about things that a mesh. Basically, in CSG, to &nbsp;&nbsp;&nbsp;store a cube with a hole in it, it would be stored as a cube minus a cylinder, whereas a mesh is just the set of points that make up that cube with a hole in it

### &nbsp;&nbsp;2. Initial Conditions

&nbsp;&nbsp;&nbsp;Another Very straight forward component. Things like intial temperature, Values of any intial neutron sources being emitted into the problem.

### &nbsp;&nbsp;3. Materials

&nbsp;&nbsp;&nbsp;This is where the magic really happens. Materials are what OpenMC uses to store any and all material properties of the model. They are assigned to a given geometry, as we will see &nbsp;&nbsp;&nbsp;later. They store anything from density, to cross sections, to crystal structure.

# &nbsp;B. Layout


&nbsp;&nbsp;How are OpenMC model definitions laid out in the Python API?

### &nbsp;&nbsp;1. Define Materials

&nbsp;&nbsp;&nbsp;Any needed materials should be defined first, with all needed properties assigned.

### &nbsp;&nbsp;2. Outline the Model

&nbsp;&nbsp;&nbsp;Now setup the boundaries of the geometry with simple shapes. 

&nbsp;&nbsp;&nbsp;As an example, let's consider a mono-block reactor core from a very simple angle:<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;1. The containment is a giant cylinder, so lets put a large cylinder parallel to the z-axis (called a zcylinder in openmc). Also, make another one but slightly bigger<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;2. There are coolant channels running through the fuel, so let's define several more zcylinders across the inside of the containment cylinder<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;3. We also need a lower and upper bound, lets define two circles at the top and bottom of the containment<br>


### &nbsp;&nbsp;3. Fill in the Model

&nbsp;&nbsp;&nbsp;Now we can fill in between boundaries with our materials<br>
&nbsp;&nbsp;&nbsp;Following that previous example, let's continue with it. For this part, it may seem confusing now, but it will make sense later<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;1. Fill between the inside of containment boundary, the outside of the coolant channel bounds, and the top and bottom circles with fuel<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;2. Fill inside coolant channel with coolant<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;3. Fill between the outside of the containment inner bound, and the inside of the larger containment bound<br>

### &nbsp;&nbsp;&nbsp;4. Setup Model to Run

&nbsp;&nbsp;&nbsp;Now we need to begin setting the model parameters<br>
&nbsp;&nbsp;&nbsp;This looks like: initial conditions, what kind of simulation, how many particles, How to split up the simulation, etc.